# Exception Handling — Safety Nets for Expected Problems 🥅

## Goal

A syntax error means Python cannot read the program. An exception happens while valid code runs. Handle failures you expect and know how to recover from; let unknown programming bugs remain visible.

## 1. `try` and specific `except`

Put only risky code in `try`. Python jumps to the first matching handler. Capture `as error` to inspect the message.

In [1]:
try:
    result=1/0
except ZeroDivisionError as error:
    print(type(error).__name__,error)

try:
    print(unknown_name)
except NameError as error:
    print('The variable has not been assigned:',error)

ZeroDivisionError division by zero
The variable has not been assigned: name 'unknown_name' is not defined


## 2. Multiple handlers

Order specific exceptions before broad `Exception`. Interactive input is simulated so Run All never pauses.

In [2]:
def divide_text(text):
    try:
        number=int(text)
        return 10/number
    except ValueError:
        return 'This is not a valid integer'
    except ZeroDivisionError:
        return 'Enter a denominator greater than 0'

for sample in ['2','hello','0']:
    print(sample,'->',divide_text(sample))

2 -> 5.0
hello -> This is not a valid integer
0 -> Enter a denominator greater than 0


## 3. `else` and `finally`

`else` runs only when `try` succeeds. `finally` always runs, even after `return` or another exception, so use it for necessary cleanup—not normal results.

In [3]:
def demonstrate(text):
    try:
        result=10/int(text)
    except (ValueError,ZeroDivisionError) as error:
        print('Problem:',error)
    else:
        print('Result:',result)
    finally:
        print('Execution complete')

demonstrate('2'); demonstrate('0')

Result: 5.0
Execution complete
Problem: division by zero
Execution complete


## 4. Common exception types

| Exception | Meaning |
|---|---|
| `ValueError` | right kind, unacceptable value |
| `TypeError` | wrong kind or unsupported operation |
| `IndexError` | sequence position missing |
| `KeyError` | mapping key missing |
| `ZeroDivisionError` | zero divisor |
| `FileNotFoundError` | path missing |
| `PermissionError` | operation not permitted |

All usually inherit from `Exception`; do not catch `BaseException` because it includes exit/interrupt signals.

In [4]:
examples=[]
for action in [lambda:int('x'),lambda:'a'+1,lambda:[1][9],lambda:{'a':1}['b'],lambda:1/0]:
    try: action()
    except Exception as error: examples.append(type(error).__name__)
print(examples)

['ValueError', 'TypeError', 'IndexError', 'KeyError', 'ZeroDivisionError']


## 5. `raise`, chaining, and custom exceptions

Raise when a function contract is broken. Custom exceptions name a domain problem. `raise NewError(...) from error` preserves the cause.

In [5]:
class InvalidAgeError(ValueError):
    pass

def set_age(age):
    if not isinstance(age,int): raise TypeError('age must be an integer')
    if not 0<=age<=130: raise InvalidAgeError('age must be between 0 and 130')
    return age

try: set_age(-5)
except InvalidAgeError as error: print(type(error).__name__,error)

InvalidAgeError age must be between 0 and 130


## 6. Files and context managers

`with` closes the file automatically. This replaces fragile manual checks like `if 'file' in locals() or not file.closed`, whose `or` can still access an undefined name.

In [6]:
from pathlib import Path
try:
    with Path('missing.txt').open(encoding='utf-8') as file:
        print(file.read())
except FileNotFoundError as error:
    print('Missing:',error.filename)

Missing: missing.txt


## 7. Nested handling and re-raising

Inner code should handle only what it can fix. Bare `raise` re-raises the current exception with its traceback.

In [7]:
def parse_positive(text):
    try: value=int(text)
    except ValueError as error: raise ValueError(f'{text!r} is not an integer') from error
    if value<=0: raise ValueError('number must be positive')
    return value

try: print(parse_positive('oops'))
except ValueError as error: print(error); print('cause:',type(error.__cause__).__name__)

'oops' is not an integer
cause: ValueError


## 8. Assertions and defensive programming

`assert condition, message` documents an internal assumption and raises `AssertionError`. Python can disable assertions, so never use them for user input, permissions, money, or required validation.

In [8]:
def average(values):
    assert values,'internal caller promised a non-empty sequence'
    return sum(values)/len(values)
print(average([2,4,6]))

4.0


## 9. When not to catch

Do not catch an exception merely to ignore it or return a misleading default. Catch at a level that can recover, add context, retry safely, or communicate clearly. Log or re-raise unexpected failures; never use a broad handler around an entire program.

## Key concepts summary

Exceptions separate normal results from failure paths. Catch narrowly, recover deliberately, and raise meaningful contract errors.

## Important syntax and quick revision cheat sheet

| Need | Syntax |
|---|---|
| Handle | `try: ... except ValueError as error:` |
| Success-only | `else:` |
| Always cleanup | `finally:` |
| Raise | `raise ValueError("message")` |
| Chain | `raise NewError(...) from error` |
| Custom | `class MyError(Exception): ...` |

## Common mistakes and interview tips

- Avoid bare `except:` and silent `pass`.
- Never catch `BaseException` in ordinary code.
- Keep `try` blocks small.
- Do not expose sensitive data in error messages.
- Assertions are not input validation.

**Revision habit:** explain what each line does, predict the result, run it, and test one edge case.